# Golden Dataset Generation Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Model:** Gemini 2.5 Flash (`gemini-2.5-flash`)

---

### What this notebook does
Generates a 50-query evaluation dataset with Gemini 2.5 Flash reference answers
grounded in KB context. The golden dataset is the ground truth used to evaluate
all three retrieval pipelines via RAGAS metrics.

### Architecture
All logic lives in `src/golden_dataset/`. This notebook imports and calls those
functions visibly — thin orchestration, no implementation.

### Model Independence
| Stage | Model | Provider |
|-------|-------|----------|
| KB Generation | `gemini-3-flash-preview` | Google AI |
| Golden Dataset | `gemini-2.5-flash` | Google AI |
| RAG Generation | `llama-3.3-70b-versatile` | Groq |

### Query Distribution
| Source | Count |
|--------|-------|
| GEFCom | 20 |
| Household | 18 |
| Cross-scale | 12 |
| **Total** | **50** |

---
## Cell 1 — Imports and Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# ── Configuration ────────────────────────────────────────────────────────────
from config import PATHS, GOLDEN_MODEL_NAME
from config.paths import create_all_directories

# ── Utilities ────────────────────────────────────────────────────────────────
from src.utils import setup_logger, log_section

# ── Golden dataset pipeline ───────────────────────────────────────────────────
from src.golden_dataset import (
    load_kb_summaries,
    select_kb_context,
    select_cross_scale_context,
    configure_gemini_golden,
    generate_golden_dataset,
    build_combined_golden_dataset,
    GEFCOM_QUERIES,
    HOUSEHOLD_QUERIES,
    CROSS_SCALE_QUERIES,
)

logger = setup_logger("golden_dataset")
create_all_directories()

logger.info("All imports successful.")
logger.info("Model: %s", GOLDEN_MODEL_NAME)
logger.info(
    "Query counts — GEFCom: %d | Household: %d | Cross-scale: %d | Total: %d",
    len(GEFCOM_QUERIES), len(HOUSEHOLD_QUERIES), len(CROSS_SCALE_QUERIES),
    len(GEFCOM_QUERIES) + len(HOUSEHOLD_QUERIES) + len(CROSS_SCALE_QUERIES),
)

---
## Cell 2 — Configure Gemini Client

Uses `gemini-2.5-flash` — different generation from the KB model (`gemini-3-flash-preview`).

In [ ]:
gemini_golden_client = configure_gemini_golden()

---
## Cell 3 — Load Knowledge Base Summaries

Loads `combined_master_summaries.csv` produced by `notebooks/01_kb_generation.ipynb`.

In [ ]:
log_section("Loading Knowledge Base summaries", 1, 4)

master_kb    = load_kb_summaries(PATHS["summaries_csv"])
gefcom_kb    = load_kb_summaries(PATHS["summaries_csv"], dataset_filter="gefcom")
household_kb = load_kb_summaries(PATHS["summaries_csv"], dataset_filter="household")

logger.info(
    "KB loaded — GEFCom: %d | Household: %d | Total: %d",
    len(gefcom_kb), len(household_kb), len(master_kb),
)

# Quick preview
master_kb[["kb_id", "dataset", "granularity", "summary"]].head(3)

---
## Cell 4 — Generate Golden Datasets

Each `generate_golden_dataset()` call is **idempotent** — already-completed golden_ids
are skipped automatically. Safe to re-run after interruption.

Estimated runtime: ~8 minutes (50 queries × ~9s average).

In [ ]:
log_section("Generating golden datasets", 2, 4)

# ── GEFCom golden dataset (20 queries) ───────────────────────────────────────
logger.info("=" * 55)
logger.info("Generating GEFCom golden dataset (20 queries)")
logger.info("=" * 55)
gefcom_golden_df = generate_golden_dataset(
    client=gemini_golden_client,
    query_list=GEFCOM_QUERIES,
    kb_df=gefcom_kb,
    dataset_source="gefcom",
    output_path=PATHS["golden_dataset"] / "gefcom_golden_dataset.csv",
    context_selector_fn=select_kb_context,
)
logger.info(
    "GEFCom golden complete: %d entries.", len(gefcom_golden_df)
)

# ── Household golden dataset (18 queries) ────────────────────────────────────
logger.info("=" * 55)
logger.info("Generating Household golden dataset (18 queries)")
logger.info("=" * 55)
household_golden_df = generate_golden_dataset(
    client=gemini_golden_client,
    query_list=HOUSEHOLD_QUERIES,
    kb_df=household_kb,
    dataset_source="household",
    output_path=PATHS["golden_dataset"] / "household_golden_dataset.csv",
    context_selector_fn=select_kb_context,
)
logger.info(
    "Household golden complete: %d entries.", len(household_golden_df)
)

# ── Cross-scale golden dataset (12 queries) ───────────────────────────────────
logger.info("=" * 55)
logger.info("Generating Cross-scale golden dataset (12 queries)")
logger.info("=" * 55)

# Cross-scale queries need the full master KB (both datasets)
# A wrapper is used so generate_golden_dataset() can call it uniformly
def cross_scale_selector(query_meta, kb_df):
    return select_cross_scale_context(query_meta, master_kb_df=kb_df)

cross_scale_golden_df = generate_golden_dataset(
    client=gemini_golden_client,
    query_list=CROSS_SCALE_QUERIES,
    kb_df=master_kb,  # Full master KB for cross-scale context
    dataset_source="cross_scale",
    output_path=PATHS["golden_dataset"] / "cross_scale_golden_dataset.csv",
    context_selector_fn=cross_scale_selector,
)
logger.info(
    "Cross-scale golden complete: %d entries.", len(cross_scale_golden_df)
)

logger.info("All golden datasets generated.")

---
## Cell 5 — Build Combined Master Golden Dataset

In [ ]:
log_section("Building combined golden dataset", 3, 4)

combined_golden = build_combined_golden_dataset(
    golden_dfs=[gefcom_golden_df, household_golden_df, cross_scale_golden_df],
    output_path=PATHS["golden_dataset"] / "combined_golden_dataset.csv",
)

logger.info("Combined golden dataset: %d total queries.", len(combined_golden))

---
## Cell 6 — Validation Report

In [ ]:
log_section("Validation report", 4, 4)

print("\n" + "=" * 65)
print("  GOLDEN DATASET — VALIDATION REPORT")
print("=" * 65)

if combined_golden.empty:
    print("  Golden dataset is empty — check logs for errors.")
else:
    print(f"\n  Total queries : {len(combined_golden)}")
    print(f"  Generator     : {combined_golden['generated_by'].iloc[0]}")

    # Missing value check
    critical_cols = [
        "user_query", "reference_answer", "expected_summary_ids",
        "expected_primary_summary_id", "expected_context_summary",
    ]
    print("\n  Missing value check:")
    for col in critical_cols:
        missing = combined_golden[col].isna().sum()
        status  = "OK" if missing == 0 else f"WARNING — {missing} missing"
        print(f"    {col:<40} {status}")

    print("\n  By dataset_source:")
    print(combined_golden["dataset_source"].value_counts().to_string())

    print("\n  By query_type:")
    print(combined_golden["query_type"].value_counts().to_string())

    print("\n  By difficulty_level:")
    print(combined_golden["difficulty_level"].value_counts().to_string())

    print("\n  By retrieval_strategy_target:")
    print(combined_golden["retrieval_strategy_target"].value_counts().to_string())

    combined_golden["answer_words"] = (
        combined_golden["reference_answer"].str.split().str.len()
    )
    print(f"\n  Reference answer length: "
          f"mean={combined_golden['answer_words'].mean():.0f} words | "
          f"min={combined_golden['answer_words'].min()} | "
          f"max={combined_golden['answer_words'].max()}")

    # Sample entries
    print("\n" + "-" * 65)
    print("  SAMPLE ENTRIES")
    print("-" * 65)
    for _, row in combined_golden.sample(
        n=min(2, len(combined_golden)), random_state=7
    ).iterrows():
        print(f"\n  [{row['dataset_source']} / {row['query_type']} / "
              f"{row['difficulty_level']}]")
        print(f"  Query  : {row['user_query']}")
        answer_preview = str(row['reference_answer'])[:300]
        ellipsis = "..." if len(str(row['reference_answer'])) > 300 else ""
        print(f"  Answer : {answer_preview}{ellipsis}")
        print(f"  By     : {row['generated_by']} @ {row['generated_at']}")

print("\n" + "=" * 65)

---
## Pipeline Complete ✅

Golden dataset saved to:
```
outputs/golden_dataset/combined_golden_dataset.csv    ← use this for evaluation
outputs/golden_dataset/gefcom_golden_dataset.csv
outputs/golden_dataset/household_golden_dataset.csv
outputs/golden_dataset/cross_scale_golden_dataset.csv
```

### Next Phase
Move to `notebooks/03_embedding_indexing.ipynb` to build the FAISS and ChromaDB
vector indexes using LangChain `CSVLoader` and `HuggingFaceEmbeddings`.